# BiTro ST Pipeline (Notebook)

This notebook provides a configurable start-to-end workflow for spatial transcriptomics in BiTro.
Edit the config cell first, then run cells step by step.

In [1]:
import os
import subprocess
from pathlib import Path

def run_cmd(cmd, env=None, cwd=None):
    print(f"\n[RUN] {' '.join(cmd)}")
    if env is not None:
        debug_keys = [
            'OUTPUT_DIR', 'HEST_DATA_DIR', 'SPATIAL_FEATURE_DIR', 'SPATIAL_GRAPH_DIR',
            'GENE_FILE', 'SAMPLE_IDS', 'CV_MODE', 'CV_HELDOUT', 'CUDA_DEVICE_ID',
            'NUMBA_CACHE_DIR', 'USE_TRANSFER_LEARNING', 'FREEZE_BACKBONE'
        ]
        print('[ENV]')
        for key in debug_keys:
            if key in env:
                print(f'  {key}={env[key]}')
    result = subprocess.run(cmd, env=env, cwd=cwd, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


In [2]:
# =========================
# Editable configuration
# =========================
PROJECT_ROOT = Path('.').resolve()

HEST_DATA_DIR = Path('/data/yujk/hovernet2feature/BiTro/demo_data/HEST')
SPATIAL_FEATURE_DIR = Path('/data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial')
SPATIAL_GRAPH_DIR = Path('/data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial')
DINOV3_WEIGHTS = Path('/data/yujk/hovernet2feature/BiTro/demo_data/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')
GENE_FILE = Path('/data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt')

# Set SAMPLE_IDS = None to process all discovered samples
SAMPLE_IDS = ['SPA148', 'SPA147']

# Feature extraction settings
DINO_BATCH_SIZE = 256
CELL_BATCH_SIZE = 50000
NUM_WORKERS = 8
N_CLUSTERS = 7
INTER_SPOT_K = 6

# Training settings
TRAIN_OUTPUT_DIR = PROJECT_ROOT / 'log_normalized_notebook'
CV_MODE = 'loo'  # 'kfold' or 'loo'
CV_HELDOUT = None  # e.g. 'TENX147' when CV_MODE='loo'
USE_TRANSFER_LEARNING = False
FREEZE_BACKBONE = False

# NOTE: transfer learning weights can be overridden with BULK_MODEL_PATH in the training env.
# The default path is still defined inside spitial_model/train.py.

print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /data/yujk/hovernet2feature/BiTro


In [3]:
required_paths = [
    HEST_DATA_DIR,
    DINOV3_WEIGHTS,
    GENE_FILE,
    PROJECT_ROOT / 'utils' / 'extract_spatial_features_dinov3.py',
    PROJECT_ROOT / 'utils' / 'cluster_kmeans_features.py',
    PROJECT_ROOT / 'spitial_model' / 'train.py',
]
for p in required_paths:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

st_dir = HEST_DATA_DIR / 'st'
all_samples = sorted([x.stem for x in st_dir.glob('*.h5ad')]) if st_dir.exists() else []
selected_samples = all_samples if SAMPLE_IDS is None else [s for s in SAMPLE_IDS if s in all_samples]
print(f'All samples: {len(all_samples)}')
print(f'Selected samples: {len(selected_samples)}')
print(selected_samples[:20])

/data/yujk/hovernet2feature/BiTro/demo_data/HEST: OK
/data/yujk/hovernet2feature/BiTro/demo_data/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth: OK
/data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt: OK
/data/yujk/hovernet2feature/BiTro/utils/extract_spatial_features_dinov3.py: OK
/data/yujk/hovernet2feature/BiTro/utils/cluster_kmeans_features.py: OK
/data/yujk/hovernet2feature/BiTro/spitial_model/train.py: OK
All samples: 2
Selected samples: 2
['SPA148', 'SPA147']


In [4]:
# Step 1: extract per-cell DINOv3 features (with per-sample PCA)
import torch
from utils.extract_spatial_features_dinov3 import (
    HESTCellFeatureExtractor,
    get_expected_num_cells,
    is_sample_completed,
)

SPATIAL_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

extractor = HESTCellFeatureExtractor(
    hest_data_dir=str(HEST_DATA_DIR),
    output_dir=str(SPATIAL_FEATURE_DIR),
    dinov3_model_path=str(DINOV3_WEIGHTS),
    bulk_pca_model_path=None,
    device=device,
    dino_batch_size=DINO_BATCH_SIZE,
    cell_batch_size=CELL_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    assign_spot=True,
)

for sid in selected_samples:
    expected = get_expected_num_cells(str(HEST_DATA_DIR), sid)
    if is_sample_completed(str(SPATIAL_FEATURE_DIR), sid, expected):
        print(f'[SKIP] {sid} already completed')
        continue
    print(f'\n[PROCESS] {sid}')
    _ = extractor.process_sample_with_independent_pca(sid)

✓ staintools available (Vahadane normalization can be enabled)
Using HuggingFace endpoint: https://hf-mirror.com
✓ DINOv3 available
Device: cuda
Initializing DINOv3 model...
Using local DINOv3 weights: /data/yujk/hovernet2feature/BiTro/demo_data/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Using DINOv3 repository: /data/yujk/hovernet2feature/dinov3
Loading DINOv3 ViT-L/16 via torch.hub...
✓ Loaded DINOv3 successfully via torch.hub
✓ Image preprocessor configured
✓ DINOv3 model ready. Feature dim: 1024
Extractor configuration:
  - DINOv3 batch size: 256
  - Cell batch size: 50000
  - Workers: 8
  - Device: cuda
  - Final feature dim: 128 (DINOv3 + PCA)
  - Backbone: DINOv3-L
⚠️  Sample SPA148 has incomplete files: Corrupted/unreadable file: Object arrays cannot be loaded when allow_pickle=False

[PROCESS] SPA148

=== Processing spatial sample: SPA148 ===
Each sample trains its own PCA model with 128-dim DINOv3 features
Loading sample data: SPA148
  WSI: /data/yujk/hovernet2featu

Batch-parallel patch extraction: 100%|██████████| 33597/33597 [00:11<00:00, 2806.77it/s]


  Batch 1 completed; extracted 33597 patches
  Current GPU memory usage: 1.13 GB
  Successfully extracted 33597 real cell patches from the WSI
Extracting DINOv3 features...
Starting extraction of 33597 patches with DINOv3 features...
Using batch size: 256 (optimized for much better GPU utilization)


DINOv3 feature extraction:   0%|          | 0/132 [00:00<?, ?it/s]

  Resource usage: CPU: 2.0%, RAM: 9.0%, GPU: 6.3% (1.48/23.5GB)


...

DINOv3 feature extraction:  95%|█████████▌| 101/106 [01:12<00:03,  1.32it/s]

  Resource usage: CPU: 1.9%, RAM: 9.1%, GPU: 6.3% (1.48/23.5GB)


DINOv3 feature extraction: 100%|██████████| 106/106 [01:16<00:00,  1.39it/s]

  Resource usage: CPU: 0.5%, RAM: 9.1%, GPU: 6.0% (1.41/23.5GB)
DINOv3 feature extraction completed: (27081, 1024)
For sample SPA147, training PCA dimensionality reduction independently...
For sample SPA147, training an independent PCA reducer...
  Checking data quality...


  Diagnostics: number of DINO feature columns with std equal to or near 0: 0/1024
  Diagnostics: column std percentiles: [1.07474087e-04 5.13317532e-02 6.06899220e-02 8.43179934e-02
 1.23405743e-01 1.87259972e-01 3.13174197e-01 4.40282600e-01
 7.17918813e-01]
  Diagnostics: in 2000 random samples, unique-row ratio: 0.089
  Diagnostics: overall DINO feature variance: 1.883099e-01
  Diagnostics: saved the first 10 DINO features to: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial/SPA147_dino_features_preview.npy
Sample SPA147 PCA training completed:
  - Input dimension: 1024
  - Output dimension: 128
  - Total explained variance ratio: 0.9847 (98.47%)
  - Explained variance ratio of the first 10 principal components:
    PC1: 0.8412 (84.12%)
    PC2: 0.0273 (2.73%)
    PC3: 0.0138 (1.38%)
    PC4: 0.0097 (0.97%)
    PC5: 0.0084 (0.84%)
    PC6: 0.0057 (0.57%)
    PC7: 0.0050 (0.50%)
    PC8: 0.0047 (0.47%)
    PC9: 0.0035 (0.35%)
    PC10: 0.0035 (0.35%)
  - Cumulative explain

In [5]:
# Step 2: add KMeans cluster labels into each .npz feature file
from utils.cluster_kmeans_features import process_one_file

for sid in selected_samples:
    npz_path = SPATIAL_FEATURE_DIR / f'{sid}_combined_features.npz'
    if not npz_path.exists():
        print(f'[MISS] {npz_path}')
        continue
    process_one_file(str(npz_path), n_clusters=N_CLUSTERS, backup=True)

Processing: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial/SPA148_combined_features.npz
[OK] Wrote back: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial/SPA148_combined_features.npz
Processing: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial/SPA147_combined_features.npz
[OK] Wrote back: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial/SPA147_combined_features.npz


In [6]:
# Step 3: construct spatial graphs
from utils.spatial_graph_construction import HESTDirectReader

SPATIAL_GRAPH_DIR.mkdir(parents=True, exist_ok=True)
builder = HESTDirectReader(
    hest_data_dir=str(HEST_DATA_DIR),
    sample_ids=selected_samples,
    features_dir=str(SPATIAL_FEATURE_DIR),
    inter_spot_k_neighbors=INTER_SPOT_K,
)
builder.load_sample_data()
builder.process_all_samples()
metadata = builder.save_graphs(str(SPATIAL_GRAPH_DIR))
print('Graph samples:', len(metadata))

=== Loading HEST data files directly ===
Loading sample: SPA148
  - AnnData: 295 spots × 15109 genes
  - Metadata: Breast tissue
  - Cell segmentation: 33597 cells
  - Image patches: available
Loading sample: SPA147
  - AnnData: 270 spots × 15290 genes
  - Metadata: Breast tissue
  - Cell segmentation: 27081 cells
  - Image patches: available
Successfully loaded 2 samples
=== Loading deep feature files ===
Loading sample SPA148 deep features...
  - Feature shape: (33597, 128)
  - Feature dimension: 128
  - Number of cells: 33597
  - Coordinates available: (33597, 2)
Loading sample SPA147 deep features...
  - Feature shape: (27081, 128)
  - Feature dimension: 128
  - Number of cells: 27081
  - Coordinates available: (27081, 2)
Successfully loaded deep features for 2 samples
=== Processing all samples ===

Processing sample: SPA148
Extracting features for sample SPA148...
  Using positions from the NPZ file as cell coordinates; aligned length: 33597
For sample SPA148, assigning cells to 

  Batch 1: 100%|██████████| 33597/33597 [00:01<00:00, 17801.45it/s]


  Successfully assigned: 7086/33597 cells
  Pixel size: 0.6936657966779244 μm/pixel
  Distance statistics: min=1.2px,, max=5370.2px,, mean=115.8px
  Distance statistics (μm): min=0.8μm,, max=3725.1μm,, mean=80.3μm
  Cells per spot: mean=24.5,, range=[1-118]
  Sample SPA148 processed successfully

Processing sample: SPA147
Extracting features for sample SPA147...
  Using positions from the NPZ file as cell coordinates; aligned length: 27081
For sample SPA147, assigning cells to spots...
  Spot coordinate range: X[266.3, 6971.4], Y[3787.1, 9024.3]
   Cell coordinate range: X[3.2, 8436.6], Y[3598.3, 9582.5]
  Assigning 27081 cells to 270 spots...
  Processing batch 1/1 (27081 files)


  Batch 1: 100%|██████████| 27081/27081 [00:01<00:00, 18109.81it/s]


  Successfully assigned: 5409/27081 cells
  Pixel size: 0.6926184269391435 μm/pixel
  Distance statistics: min=1.1px,, max=2417.7px,, mean=117.7px
  Distance statistics (μm): min=0.7μm,, max=1674.6μm,, mean=81.5μm
  Cells per spot: mean=20.4,, range=[1-101]
  Sample SPA147 processed successfully
=== Saving graph structures ===

Saving sample SPA148...
Building sample SPA148 intra-spot graphs...


Building intra-spot graphs: 100%|██████████| 289/289 [00:00<00:00, 556.36it/s]


  Built 289 intra-spot graphs, each using 8-NN connections
Building sample SPA148 inter-spot graph...
  Built inter-spot graph: 295 spots, 3540 edges

Saving sample SPA147...
Building sample SPA147 intra-spot graphs...


Building intra-spot graphs: 100%|██████████| 265/265 [00:00<00:00, 607.37it/s]


  Built 265 intra-spot graphs, each using 8-NN connections
Building sample SPA147 inter-spot graph...
  Built inter-spot graph: 270 spots, 3240 edges

Graph data saved to: /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial
- Intra-spot graphs: /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial/hest_intra_spot_graphs.pkl
- Inter-spot graph: /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial/hest_inter_spot_graphs.pkl
- Metadata: /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial/hest_graph_metadata.json
- Processed data: /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial/hest_processed_data.pkl
Graph samples: 2


In [7]:
# Step 4: train spatial model
env = os.environ.copy()
env['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'

train_cmd = [
    'python', 'spitial_model/train.py',
    '--output_dir', str(TRAIN_OUTPUT_DIR),
    '--hest_data_dir', str(HEST_DATA_DIR),
    '--features_dir', str(SPATIAL_FEATURE_DIR),
    '--graph_dir', str(SPATIAL_GRAPH_DIR),
    '--gene_file', str(GENE_FILE),
    '--sample_ids', ','.join(selected_samples),
    '--cv_mode', CV_MODE,
    '--cuda_device_id', '0',
    '--use_transfer_learning', str(USE_TRANSFER_LEARNING).lower(),
    '--freeze_backbone', str(FREEZE_BACKBONE).lower(),
    '--numba_cache_dir', env['NUMBA_CACHE_DIR'],
]
if CV_HELDOUT:
    train_cmd.extend(['--cv_heldout', CV_HELDOUT])

print('Step 4 configuration:')
print('  HEST_DATA_DIR =', HEST_DATA_DIR)
print('  SPATIAL_FEATURE_DIR =', SPATIAL_FEATURE_DIR)
print('  SPATIAL_GRAPH_DIR =', SPATIAL_GRAPH_DIR)
print('  selected_samples =', selected_samples)
print('  CV_MODE =', CV_MODE)
print('  CV_HELDOUT =', CV_HELDOUT)

run_cmd(train_cmd, env=env, cwd=str(PROJECT_ROOT))

Step 4 configuration:
  HEST_DATA_DIR = /data/yujk/hovernet2feature/BiTro/demo_data/HEST
  SPATIAL_FEATURE_DIR = /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial
  SPATIAL_GRAPH_DIR = /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial
  selected_samples = ['SPA148', 'SPA147']
  CV_MODE = loo
  CV_HELDOUT = None

[RUN] python spitial_model/train.py --output_dir /data/yujk/hovernet2feature/BiTro/log_normalized_notebook --hest_data_dir /data/yujk/hovernet2feature/BiTro/demo_data/HEST --features_dir /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial --graph_dir /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial --gene_file /data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt --sample_ids SPA148,SPA147 --cv_mode loo --cuda_device_id 0 --use_transfer_learning false --freeze_backbone false --numba_cache_dir /tmp/numba_cache
[ENV]
  NUMBA_CACHE_DIR=/tmp/numba_cache
=== HEST Spatial Supervised Training ===
✓ Using direct file reading (no HEST API require

Epoch 1/10:   0%|                                                             | 0/3 [00:00<?, ?it/s]

Debug - Batch 0:
  Predictions shape: torch.Size([128, 785]), range: [0.3668, 1.1043]
  Targets shape: torch.Size([128, 785]), range: [-1.6672, 11.9911]
  predictions.requires_grad: True


=== Evaluating model ===


Evaluating: 100%|██████████| 3/3 [00:00<00:00,  5.82it/s]


...

Evaluating: 100%|██████████| 3/3 [00:00<00:00,  6.47it/s]


Evaluation samples: 270 spots × 785 genes

=== Evaluation Results ===
Overall MSE: 0.267963
Overall Correlation: 0.434689 (p=0.00e+00)
Mean Gene Correlation: 0.171322
Median Gene Correlation: 0.177762
Std Gene Correlation: 0.066207
Mean Spot Correlation: 0.511211
Median Spot Correlation: 0.540335
Mean Gene MSE: 0.267963
Mean Spot MSE: 0.267963
Epoch 10/10
  Train Loss: 79.169837, Test Loss: 1.250684
  Mean Gene Pearson: 0.171322
  Overall Pearson: 0.434689
  LR: 1.00e-06
  Grad Norm: 5.000000
  *** Saving best model (Test Loss: 1.250684, Epoch: 10) ***

=== Training completed ===
Best test loss: 1.250684 (Epoch 10)
Total epochs: 10
Loaded best model weights for evaluation
=== Evaluating model ===


Evaluating: 100%|██████████| 3/3 [00:00<00:00,  6.44it/s]


Evaluation samples: 270 spots × 785 genes

=== Evaluation Results ===
Overall MSE: 0.267963
Overall Correlation: 0.434689 (p=0.00e+00)
Mean Gene Correlation: 0.171322
Median Gene Correlation: 0.177761
Std Gene Correlation: 0.066206
Mean Spot Correlation: 0.511211
Median Spot Correlation: 0.540335
Mean Gene MSE: 0.267963
Mean Spot MSE: 0.267963
Results saved to /data/yujk/hovernet2feature/BiTro/log_normalized_notebook
✓ Saved per-epoch metrics to /data/yujk/hovernet2feature/BiTro/log_normalized_notebook/fold_1_epoch_metrics.json

=== Fold 2 Completed ===
Final test loss: 1.2506840030352275
Overall correlation: 0.434689
Mean gene correlation: 0.171322
Std gene correlation: 0.066206

LEAVE-ONE-OUT CROSS VALIDATION COMPLETED
Average overall correlation: 0.478763 ± 0.044074
Average gene correlation: 0.174575 ± 0.003252
Average final test loss: 1.120035 ± 0.130649
Average BEST overall correlation: 0.479291 ± 0.043546
Average BEST gene correlation: 0.184810 ± 0.006984

Final results saved to:

CompletedProcess(args=['python', 'spitial_model/train.py', '--output_dir', '/data/yujk/hovernet2feature/BiTro/log_normalized_notebook', '--hest_data_dir', '/data/yujk/hovernet2feature/BiTro/demo_data/HEST', '--features_dir', '/data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial', '--graph_dir', '/data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial', '--gene_file', '/data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt', '--sample_ids', 'SPA148,SPA147', '--cv_mode', 'loo', '--cuda_device_id', '0', '--use_transfer_learning', 'false', '--freeze_backbone', 'false', '--numba_cache_dir', '/tmp/numba_cache'], returncode=0)

In [8]:
# Step 5: quick artifact check
print('Spatial feature dir exists:', SPATIAL_FEATURE_DIR.exists())
print('Spatial graph dir exists:', SPATIAL_GRAPH_DIR.exists())
print('Train output dir exists:', TRAIN_OUTPUT_DIR.exists())

if SPATIAL_GRAPH_DIR.exists():
    for p in sorted(SPATIAL_GRAPH_DIR.glob('*')):
        print(' -', p.name)

Spatial feature dir exists: True
Spatial graph dir exists: True
Train output dir exists: True
 - .gitkeep
 - hest_graph_metadata.json
 - hest_inter_spot_graphs.pkl
 - hest_intra_spot_graphs.pkl
 - hest_processed_data.pkl
